## Описание проекта

Для продуктовой команды сервиса заказа такси необходимо настроить витрину данных, которая будет автоматически агрегировать информацию о поездках по каждому способу оплаты. 

**Задачи**<br>
Написать PySpark-скрипт, который рассчитает ключевые показатели: количество поездок, среднюю стоимость поездки, средние чаевые и суммарную выручку по каждому типу оплаты.
Создать DAG в Airflow, который будет ежедневно:

* проверять наличие новых файлов с данными;

* запускать Spark-задачу;

* формировать обновлённую итоговую таблицу `taxi_payment_summary`.

* Таблицу нужно записать в ClickHouse с помощью JDBC-драйвера.

Эта таблица станет основой для финансовых отчётов и аналитических дашбордов.

## Описание данных

Таблица `taxi_data` содержит данные об активности пользователей и состоит из следующих полей:

* `taxi_id` — идентификатор водителя;

* `trip_start_timestamp` — время начала поездки;

* `trip_end_timestamp` — время окончания поездки;

* `trip_seconds` — длительность поездки в секундах;

* `trip_miles` — дистанция поездки;

* `fare` — стоимость поездки;

* `tips` — размер чаевых;

* `trip_total` — общая стоимость поездки: стоимость поездки + чаевые + комиссия;

* `payment_type` — способ оплаты.

## Шаг 1. Настраиваем Spark-агрегацию

In [ ]:
# filename=my_spark_job.py

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import sys

# Создаём Spark-сессию
spark = SparkSession.builder.appName("myAggregateTaxi").config("fs.s3a.endpoint", "storage.yandexcloud.net").getOrCreate()

# Указываем порт и параметры кластера ClickHouse
jdbcPort = 8443
jdbcHostname = "rc1a-3jouval14nne7aun.mdb.yandexcloud.net"
username = "**************" # имя пользователя закрыто для соблюдения NDA
jdbcDatabase = "playground_" + username
jdbcUrl = f"jdbc:clickhouse://{jdbcHostname}:{jdbcPort}/{jdbcDatabase}?ssl=true"

# Считываем исходные данные
df = spark.read.parquet(f"s3a://da-plus-dags/project_04/taxi_data.parquet", inferSchema=True, header=True)

# Строим агрегат по способу оплаты
result_df = df.groupBy("payment_type").agg(
    F.count("*").alias("trip_count"),
    F.avg("fare").alias("fare_avg"),
    F.avg("tips").alias("tips_avg"),
    F.sum("trip_total").alias("trip_total_sum")
)

# Записываем результат агрегации
result_df.write.format("jdbc") \
    .option("url", jdbcUrl) \
    .option("user", username) \
    .option("password", "**************") \  # пароль пользователя закрыт для соблюдения NDA
    .option("dbtable", "taxi_data_aggregate") \
    .mode('append') \
    .save()

## Шаг 2. Настраиваем DAG

In [ ]:
# filename=bookmate_dag.py

from datetime import datetime
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

DAG_ID = "taxi_data_analysis"
user = '**************'

with DAG(
    DAG_ID,
    schedule='@daily',
    start_date=datetime(2026, 1, 1),
    catchup=False
) as dag:
    # 1) Ждём появления входного файла в S3
    wait_for_input = S3KeySensor(
        task_id="s3_sensor_for_taxi_data",
        bucket_name="da-plus-dags",
        bucket_key="project_04/taxi_data.parquet",
        aws_conn_id="s3",
        poke_interval=60*5, # проверяем 5 мин
        timeout=60*60,      # с перерывом час
        mode="poke",
        wildcard_match=False
    )
    # 2) Запускаем PySpark-задание на кластере Dataproc (оператор Яндекс Облака)
    run_pyspark = DataprocCreatePysparkJobOperator(
        task_id="taxi_create_pyspark_job",
        name="taxi_aggregate_and_load",
        cluster_id="c9q4134h5vi546h1e148",
        main_python_file_uri=f"s3a://da-plus-dags/{user}/jobs/my_spark_job.py"
    )
    # 3)  Зависимости 
    wait_for_input >> run_pyspark

## Шаг 3. Запускаем DAG с помощью Airflow UI

Результат полученной витрины:

|payment_type|trip_count|fare_avg  |tips_avg    |trip_total_sum|
|------------|----------|----------|------------|--------------|
|No Charge   |     12843|13.5359535|  0.12569416|      184374.6|
|Cash        |   1410155| 11.194248|0.0033251946|      16971150|
|Prcard      |       968| 11.172262|  0.25764462|      11513.15|
|Unknown     |      5180| 11.791628|  0.35713595|      67725.07|
|Credit Card |   1108843| 15.869293|    3.477157|      23160130|
|Way2ride    |         3|       3.5|  0.79333335|         14.38|
|Dispute     |      1842| 13.327953|         0.0|      27052.29|
|Pcard       |       878|  9.833428|  0.18314351|       9175.55|